# Scenario 1: Simple Chatbot Monitoring with Galileo

## What You'll Learn

This notebook walks you through monitoring a **customer support chatbot** using the **Galileo Python SDK**. By the end, you'll understand how to:

1. **Connect** your app to Galileo's observability platform
2. **Auto-instrument** OpenAI calls so every LLM interaction is logged automatically
3. **Stream** responses and still capture complete traces
4. **Group** multi-turn conversations into sessions
5. **Enable quality and safety metrics** (correctness, PII, toxicity) to score your traces

## What is Galileo?

**Galileo** is an AI observability platform. Think of it like application monitoring (Datadog, New Relic) but purpose-built for LLM applications. It helps you answer questions like:
- *"Are my chatbot's responses actually correct?"*
- *"Is the model leaking PII or producing toxic content?"*
- *"How does conversation quality change across sessions?"*

## Key Concepts

| Concept | What it means |
|---------|--------------|
| **Project** | A top-level container in Galileo — like a folder for one app or use case |
| **Log Stream** | A named feed of traces inside a project — use separate streams for dev, staging, prod |
| **Trace** | One end-to-end request through your app (a user question → model answer) |
| **Span** | A unit of work inside a trace (e.g., an LLM call, a tool call, a retrieval step) |
| **Session** | A group of related traces — e.g., a multi-turn conversation with one user |
| **Metrics** | Automated quality/safety scores that Galileo computes on your traces |

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY` (see `.env.example`)
- Dependencies installed via `uv sync`

## Step 0: Load Environment Variables

Before using any Galileo or OpenAI APIs, we need to load our **API keys** from a `.env` file. This is a standard practice to keep secrets out of your code.

The cell below searches for a `.env` file in the current directory or parent directory and loads it using `python-dotenv`. Your `.env` file should contain:

```
GALILEO_API_KEY=your-galileo-key-here
OPENAI_API_KEY=your-openai-key-here
```

**Where to get these keys:**
- **Galileo API Key**: Sign up at [galileo.ai](https://galileo.ai), go to Settings → API Keys
- **OpenAI API Key**: Get one from [platform.openai.com/api-keys](https://platform.openai.com/api-keys)

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


## Step 0b: Import Galileo SDK and Set Constants

Now we import the Galileo SDK modules we'll use throughout this notebook:

| Import | Purpose |
|--------|---------|
| `galileo_context` | The main entry point — initializes Galileo and manages the current logging session |
| `GalileoLogger` | For manual trace logging (used in advanced scenarios) |
| `GalileoMetrics` | Enum of all available metrics (correctness, toxicity, PII, etc.) |
| `Message`, `MessageRole` | Data classes to represent chat messages in Galileo's format |
| `openai` from `galileo.openai` | A **wrapped** OpenAI client — it works exactly like the normal `openai` library but auto-logs every call to Galileo |

We also define three constants:
- **`PROJECT_NAME`** — the Galileo project that will hold all our traces
- **`LOG_STREAM_NAME`** — the specific log stream within the project
- **`MODEL`** — which OpenAI model to use (`gpt-4o-mini` is fast and cheap for testing)

In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.openai import openai
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S1_Chatbot'
LOG_STREAM_NAME = 'chatbot-monitoring'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, MODEL

## Step 1: Initialize Galileo

This is the first real Galileo operation. `galileo_context.init()` does several things:

1. **Authenticates** with the Galileo API using your `GALILEO_API_KEY`
2. **Creates or finds** the project named `GalileoEval_S1_Chatbot`
3. **Creates or finds** the log stream named `chatbot-monitoring` inside that project
4. **Sets up auto-instrumentation** — after this call, any OpenAI calls made through Galileo's wrapped client will be automatically logged

**What's a Log Stream?** Think of it as a named channel where traces are sent. You might have one log stream for development, another for staging, and another for production. This keeps your data organized.

After initialization, we print the **console URLs** so you can open the Galileo web UI and see your traces appear in real time.

### Deep Dive: GalileoPythonConfig

`GalileoPythonConfig` is the SDK's configuration object. It reads settings from multiple sources (in order of priority):

1. **Environment variables** with prefix `GALILEO_` (e.g., `GALILEO_API_KEY` from your `.env`)
2. **Config file** `galileo-python-config.json` in `~/.galileo/`
3. **Direct attributes** set on the config object

| Attribute | Purpose |
|-----------|---------|
| `console_url` | Base URL of the Galileo web UI (used to build clickable project links) |
| `api_url` | Galileo API base URL (for custom/on-prem deployments) |
| `api_key` | Your `GALILEO_API_KEY` for authentication |
| `home_dir` | Galileo config directory (default `~/.galileo`) |

In this notebook, we mainly use `config.console_url` to construct URLs so you can click through to your project in the Galileo Console:

```python
project_url = f'{config.console_url}project/{project_id}'
log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
```

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## Step 2: Create an OpenAI Client Through Galileo

This is the **magic** of Galileo's OpenAI integration. Instead of importing `openai` the normal way:

```python
# Normal OpenAI (NOT instrumented)
from openai import OpenAI
client = OpenAI()
```

We use Galileo's wrapped version:

```python
# Galileo-instrumented OpenAI (auto-logs every call)
from galileo.openai import openai
client = openai.OpenAI()
```

**The client works identically to the normal OpenAI client** — same methods, same parameters, same responses. The only difference is that every API call is automatically captured as a trace in Galileo. You don't need to add any logging code.

This pattern is called **auto-instrumentation** — Galileo wraps the underlying HTTP calls so that traces are created transparently.

In [15]:
client = openai.OpenAI()
client


## Step 3: Log a Single Chat Completion

This is the simplest possible Galileo workflow — make one LLM call and log it:

1. **`galileo_context.init()`** — re-initializes the context (ensures a fresh trace batch)
2. **`client.chat.completions.create()`** — makes a normal OpenAI API call. Because we're using the Galileo-wrapped client, this call is automatically captured as a trace
3. **`galileo_context.flush()`** — sends all buffered traces to Galileo's servers

**Why do we need `flush()`?** Galileo buffers traces locally for performance. `flush()` pushes them to the server so they appear in the Galileo Console. Think of it like `git push` — your work exists locally until you flush.

**What gets logged?** The full trace includes:
- The input messages (system prompt + user question)
- The model's response
- Token counts, latency, and model name
- Any metrics you've enabled (we'll add those in Step 6)

After running this cell, click the log stream URL from Step 1 to see the trace in Galileo!

In [18]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent for a software company.'},
        {'role': 'user', 'content': 'How do I reset my password?'},
    ],
    max_tokens=100,
)

galileo_context.flush()
response.choices[0].message.content


'To reset your password, please follow these steps:\n\n1. **Go to the Login Page**: Visit the login page of the application or website.\n\n2. **Click on \'Forgot Password?\'**: Look for a link or button that says "Forgot Password?" or "Reset Password" and click on it.\n\n3. **Enter Your Email Address**: You will usually be prompted to enter the email address associated with your account.\n\n4. **Check Your Email**: After submitting your email,'

## Step 4: Log a Streaming Response

In production chatbots, you often use **streaming** (`stream=True`) so users see tokens appear in real time instead of waiting for the full response. But this changes how you handle the response:

**Without streaming:** You get one complete response object.
**With streaming:** You get many small **chunks**, each containing a few tokens (called a **delta**).

### How streaming works with Galileo

```
stream=True
    │
    ▼
chunk 1: "Our"
chunk 2: " business"
chunk 3: " hours"
chunk 4: " are"
...
    │
    ▼
All chunks collected → full response assembled
    │
    ▼
galileo_context.flush()  ← sends complete trace to Galileo
```

**Why flush after the stream ends?** Galileo needs the complete response to log a meaningful trace. While streaming, the reply isn't complete until you've consumed all chunks. So you:
1. Loop over chunks and collect `chunk.choices[0].delta.content`
2. Join them into the full response
3. Call `flush()` to send the complete trace

The Galileo wrapper handles most of this internally, but calling `flush()` ensures everything is sent.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent.'},
        {'role': 'user', 'content': 'What are your business hours?'},
    ],
    max_tokens=80,
    stream=True,
)

chunks = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chunks.append(chunk.choices[0].delta.content)

galileo_context.flush()
''.join(chunks)

## Step 5: Track a Multi-Turn Conversation Session

Real chatbots aren't just single-question-and-answer. Users have **conversations** — they ask follow-ups, clarify, and build on previous messages. Galileo's **sessions** let you group these related turns together.

### What is a Session?

A session is a logical grouping of traces that belong to one conversation. Without sessions, each LLM call would appear as an isolated trace in Galileo. With sessions, you can see the entire conversation thread.

### How it works

```
start_session('customer-session-001')
        │
        ▼
┌─────────────────────────────────────────────────────────────────┐
│  Turn 1: "I can't log into my account."  →  LLM responds       │
├─────────────────────────────────────────────────────────────────┤
│  Turn 2: "I tried that but it still doesn't work."  →  LLM     │
├─────────────────────────────────────────────────────────────────┤
│  Turn 3: "Okay that worked, thank you!"  →  LLM responds       │
└─────────────────────────────────────────────────────────────────┘
        │
        ▼
galileo_context.flush()   ←  sends ONE trace for the whole session
```

### Why flush once at the end?

Each `flush()` sends the accumulated traces. If you flush after every turn, you'd get separate traces instead of one cohesive conversation thread. Flushing once at the end creates a single trace containing the full multi-turn exchange.

### The conversation pattern

Notice how we maintain a `conversation` list and append both assistant replies and new user messages. This is standard OpenAI multi-turn — each call includes the full conversation history so the model has context.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
galileo_context.start_session(name='customer-session-001')

conversation = [
    {'role': 'system', 'content': 'You are a customer support agent. Be concise.'},
    {'role': 'user', 'content': "I can't log into my account."},
]

first = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)
conversation.append({'role': 'assistant', 'content': first.choices[0].message.content})
conversation.append({'role': 'user', 'content': "I tried that but it still doesn't work."})

second = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)
conversation.append({'role': 'assistant', 'content': second.choices[0].message.content})
conversation.append({'role': 'user', 'content': 'Okay that worked, thank you!'})

final_response = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)

galileo_context.flush()
final_response.choices[0].message.content

## Step 6: Enable Quality and Safety Metrics

So far, Galileo has been logging your traces (inputs, outputs, token counts, latency). But the real power of Galileo is **automated evaluation** — it can score every trace on multiple quality and safety dimensions.

### What are Metrics?

Metrics are automated scorers that Galileo runs on your traces. They fall into several categories:

| Category | Metrics in this step | What they measure |
|----------|---------------------|-------------------|
| **Quality** | `correctness` | Is the response factually accurate? |
| **Quality** | `instruction_adherence` | Does the response follow the system prompt instructions? |
| **Safety** | `input_pii` / `output_pii` | Does the input or output contain personally identifiable information (names, emails, SSNs)? |
| **Safety** | `input_toxicity` / `output_toxicity` | Does the input or output contain toxic, offensive, or harmful content? |

### How `enable_metrics()` works

- Metrics are configured **on the log stream**, not per-trace
- Once enabled, Galileo scores **all future traces** sent to that log stream
- Each `enable_metrics()` call **replaces** the entire metric set — so include all metrics you want active
- Scores appear as columns in the Galileo Console alongside your traces

### When to use these metrics

- **Correctness + Instruction Adherence**: Essential for any chatbot — catch hallucinations and off-topic responses
- **PII Detection**: Required for compliance (GDPR, HIPAA) — flag traces where users accidentally share or the model leaks personal data
- **Toxicity**: Safety net for customer-facing apps — detect harmful content before it reaches users

After running this cell, go to the Galileo Console (use the URLs printed in Step 1) to see metric scores on your traces.

In [22]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.input_pii,
        GalileoMetrics.output_pii,
        GalileoMetrics.input_toxicity,
        GalileoMetrics.output_toxicity,
    ],
)
project_url = f'{config.console_url}project/{project_id}'
log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
print(project_url)
print(log_stream_url)

https://console.demo-v2.galileocloud.io/project/979c1e2b-bbeb-4ed1-9977-9dabc0513fda
https://console.demo-v2.galileocloud.io/project/979c1e2b-bbeb-4ed1-9977-9dabc0513fda/log-streams/6daf77cf-641f-4fae-ae0e-bfcb7718a355


### Check Your Metrics in the Console

Click the log stream URL printed above to open the Galileo Console. You should see your traces with metric score columns. Try sorting by a metric to find your best and worst responses!

## Step 7: Clean Up the Demo Project

This cell deletes the Galileo project we created, including all its log streams and traces. **Only run this when you're done exploring** — once deleted, the data is gone.

In a real application, you would never delete the project. This cleanup is only here so the notebook can be re-run cleanly without accumulating duplicate projects.

In [ ]:
delete_project(name=PROJECT_NAME)
